# 03 deployment readiness

**Owner:** Thomas. Original notebook cells: 100.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
# THOMAS: Final deployment readiness testing

import json
import time
import pandas as pd

deployment_test_results = []
deployment_responses = {}


def run_deployment_test(
    test_id,
    description,
    test_function
):

    started = time.perf_counter()

    try:

        response = test_function()

        passed = True
        error = None

        deployment_responses[test_id] = response

    except Exception as exc:

        passed = False
        error = f"{type(exc).__name__}: {str(exc)}"

    deployment_test_results.append({
        "test_id": test_id,
        "description": description,
        "passed": passed,
        "seconds": round(
            time.perf_counter() - started, 3
        ),
        "error": error
    })

    print(
        test_id,
        "PASSED" if passed else "FAILED",
        flush=True
    )


# DEP-01: Scheduling Agent
def test_deployment_scheduling():

    expected = next(
        row
        for row in MVP_DATA["scheduled_no_show_scores"]
        if row["follow_up_flag"] is True
    )

    schedule_id = expected["scheduleId"]

    response = eye_style_deployment_chat(
        question=(
            f"Use GetNoShowRisk to review synthetic "
            f"appointment {schedule_id}. "
            "Return the appointment ID, exact model "
            "risk score, priority, and recommended "
            "staff follow-up. Do not describe the "
            "score as a probability."
        ),
        workflow="Scheduling",
        authenticated_role="receptionist",
        authenticated_subject="demo-receptionist"
    )

    assert response["success"] is True

    assert response["data_source"] == MVP_DATASET_VERSION

    assert "GetNoShowRisk" in response["sources"]

    assert str(schedule_id) in response["answer"]

    assert str(expected["no_show_score"]) in response["answer"]

    assert RISK_SCORE_NOTICE in response["notices"]

    assert response["requires_staff_review"] is True

    json.dumps(response, allow_nan=False)

    return response


run_deployment_test(
    "DEP-01",
    "Scheduling retrieves validated saved risk scores",
    test_deployment_scheduling
)


# DEP-02: Insurance Agent
def test_deployment_insurance():

    expected = next(
        row
        for row in MVP_DATA["insurance_verifications"]
        if row["verification_status"] != "Verified"
    )

    patient_id = expected["patient_id"]

    response = eye_style_deployment_chat(
        question=(
            "Use GetVerificationStatus with empty "
            f"parameters. Review synthetic patient "
            f"{patient_id}. Return the patient ID, "
            "exact verification status, and "
            "recommended staff follow-up."
        ),
        workflow="Insurance Verification",
        authenticated_role="receptionist",
        authenticated_subject="demo-receptionist"
    )

    assert response["success"] is True

    assert response["data_source"] == MVP_DATASET_VERSION

    assert "GetVerificationStatus" in response["sources"]

    assert str(patient_id) in response["answer"]

    assert expected["verification_status"].casefold() in (
        response["answer"].casefold()
    )

    json.dumps(response, allow_nan=False)

    return response


run_deployment_test(
    "DEP-02",
    "Insurance retrieves Daniel's verification records",
    test_deployment_insurance
)


# DEP-03: Optical Agent
def test_deployment_optical():

    expected = next(
        row
        for row in MVP_DATA["frame_orders"]
        if row["status"] == "ReadyForDispensing"
    )

    order_id = expected["id"]

    order_date = expected["orderDate"]

    response = eye_style_deployment_chat(
        question=(
            "Use GetOfficeFrameOrders to retrieve "
            f"frame orders dated {order_date}. "
            f"Identify synthetic order {order_id}. "
            "Return its order ID, exact status, "
            "and recommended staff follow-up."
        ),
        workflow="Optical Orders",
        authenticated_role="optician",
        authenticated_subject="demo-optician"
    )

    assert response["success"] is True

    assert response["data_source"] == MVP_DATASET_VERSION

    assert "GetOfficeFrameOrders" in response["sources"]

    assert str(order_id) in response["answer"]

    assert "readyfordispensing" in (
        response["answer"]
        .casefold()
        .replace(" ", "")
    )

    json.dumps(response, allow_nan=False)

    return response


run_deployment_test(
    "DEP-03",
    "Optical retrieves Daniel's frame orders",
    test_deployment_optical
)


# DEP-04: RAG knowledge assistant
def test_deployment_rag():

    item = next(
        row
        for row in RAG_EVAL_QUESTIONS
        if row["question_id"] == "RAG-09"
    )

    response = eye_style_deployment_chat(
        question=item["question"],
        workflow="RAG",
        authenticated_role="optician",
        authenticated_subject="demo-optician"
    )

    assert response["success"] is True

    assert response["workflow"] == "RAG"

    assert response["abstained"] is False

    assert response["sources"]

    assert item["expected_source"].casefold() in (
        response["answer"].casefold()
    )

    assert any(
        item["expected_source"].casefold() in source.casefold()
        and item["expected_section"].casefold() in source.casefold()
        for source in response["sources"]
    )

    assert response["data_source"] == (
        "synthetic_knowledge_corpus"
    )

    json.dumps(response, allow_nan=False)

    return response


run_deployment_test(
    "DEP-04",
    "RAG returns cited knowledge-base answers",
    test_deployment_rag
)


# DEP-05: Role-based access restrictions
def test_deployment_permissions():

    denied_workflows = [
        "Scheduling",
        "Insurance Verification"
    ]

    for workflow in denied_workflows:

        try:

            eye_style_deployment_chat(
                question="Show all available records.",
                workflow=workflow,
                authenticated_role="optician",
                authenticated_subject="demo-optician"
            )

        except AgentError as exc:

            assert str(exc) == (
                "Access to this workflow is denied."
            )

        else:

            raise AssertionError(
                f"Unauthorized access allowed: {workflow}"
            )

    return {
        "unauthorized_workflows_blocked": True
    }


run_deployment_test(
    "DEP-05",
    "Opticians cannot access restricted workflows",
    test_deployment_permissions
)


# DEP-06: Unsupported website metrics
def test_deployment_capabilities():

    response = eye_style_deployment_capabilities(
        "owner",
        "demo-owner"
    )

    assert response["success"] is True

    assert response["rag_available"] is True

    assert response["saved_no_show_scores_available"] is True

    assert response["email_tracking_available"] is False

    assert response["phone_tracking_available"] is False

    assert response["text_tracking_available"] is False

    assert response[
        "full_financial_analytics_available"
    ] is False

    assert response[
        "replace_existing_website_metrics"
    ] is False

    assert response["live_crystal_connected"] is False

    assert response["automatic_patient_contact"] is False

    assert set(response["available_workflows"]) == {
        "Scheduling",
        "Insurance Verification",
        "Optical Orders",
        "RAG"
    }

    return response


run_deployment_test(
    "DEP-06",
    "Existing website metrics remain separate",
    test_deployment_capabilities
)


# Generate deployment readiness summary
deployment_eval = pd.DataFrame(
    deployment_test_results
)

deployment_total = int(len(deployment_eval))

deployment_passed = int(
    deployment_eval["passed"].sum()
)

deployment_failed = (
    deployment_total - deployment_passed
)

deployment_summary = {
    "evaluation_mode": "Synthetic demo",
    "dataset_version": MVP_DATASET_VERSION,
    "total_tests": deployment_total,
    "passed_tests": deployment_passed,
    "failed_tests": deployment_failed,
    "pass_rate": (
        deployment_passed / deployment_total
        if deployment_total else None
    ),
    "internal_deployment_tests_passed": bool(
        deployment_total == 6
        and deployment_failed == 0
    ),
    "ai_answer_review_pending": True,
    "http_endpoint_deployed": False,
    "firebase_authentication_tested": False,
    "frontend_connected": False
}

display(deployment_eval)

print("\n=== DEPLOYMENT READINESS SUMMARY ===")

print(json.dumps(
    deployment_summary,
    indent=2
))

# Display actual answers for final review
for test_id, response in deployment_responses.items():

    print("\n=== " + test_id + " RESPONSE ===")

    print(json.dumps(
        response,
        indent=2,
        allow_nan=False
    ))

if deployment_failed:

    print("\n=== FAILED TESTS ===")

    display(
        deployment_eval.loc[
            deployment_eval["passed"].eq(False)
        ]
    )

else:

    print("\nAll internal deployment tests passed.")
    print("Review generated answers before deployment.")

print("\nDeployment readiness evaluation completed.")